# 📘 Notebook 11: Validação Retrospectiva Causal de Setembro/2026, Eventos Climáticos Extremos e Recálculo de Margens de Segurança

**Programa de Mestrado Profissional**  
**Projeto:** Sistema Preditivo de Apoio à Decisão para Demanda de Refeições no Restaurante Institucional (IFSP Itaquaquecetuba)  
**Fase:** Preparação da Dissertação Definitiva (Banca de Defesa)  
**Autor:** Michel  
**Artefato Avaliado:** Validação Cega *Out-of-Time* (Setembro/2026) e Política de Mitigação de Riscos Operacionais.

---

### 🎯 Objetivos Metodológicos deste Caderno:
1. **Auditar o Teste Retrospectivo Cego de Setembro/2026:** Analisar o comportamento do modelo em 16 dias com funcionamento de refeitório, separando dias normais de dias com eventos atípicos de enchente.
2. **Avaliar a Resiliência Algorítmica vs. Humana em Eventos Extremos:** Investigar por que ocorreram dias com apenas 14 e 23 refeições e comparar o erro da IA contra o planejamento da Nutricionista institucional.
3. **Recalcular as Margens de Segurança (Conformal Prediction & Newsvendor):**
   - Comparar o impacto da contaminação de resíduos extremos no quantil conformal.
   - Recalibrar a margem de segurança para o regime de operação padrão.
4. **Estabelecer a Regra de Mitigação Climática Operacional:** Propor uma lógica formal de governança para que o artefato (Streamlit) proteja a instituição tanto do desabastecimento quanto do desperdício em massa.
5. **Garantir a Reprodutibilidade:** Gerar as tabelas e métricas oficiais que fundamentam os novos capítulos da Dissertação de Defesa.


## 1. Configuração do Ambiente e Importação de Bibliotecas


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuração de estilo visual acadêmico para figuras
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8
sns.set_theme(style='whitegrid', font_scale=1.1)

print("✅ Bibliotecas carregadas com sucesso!")


## 2. Ingestão dos Dados de Validação de Setembro de 2026

Carregamos a base oficial de validação retrospectiva de setembro, contendo:
* O consumo real verificado na catraca;
* O status do dado (`observado` para operação normal e `atipico` para dias de crise climática/mobilidade);
* As variáveis meteorológicas locais (precipitação acumulada e horas de chuva contínua);
* As previsões autônomas dos modelos de Machine Learning (Ridge, LightGBM, CatBoost), o Modelo Híbrido e o planejamento humano da Nutricionista.


In [ ]:
# Definição dos caminhos
BASE_DIR = Path("..")
ARQ_SETEMBRO = BASE_DIR / "reports" / "previsoes_prospectivas_setembro_outubro_2026" / "tabela_completa_setembro_2026.csv"
ARQ_BRONZE = BASE_DIR / "dados" / "bronze" / "consumo" / "consumo_setembro_2026.csv"

# Carga das bases
df_set = pd.read_csv(ARQ_SETEMBRO, sep=";")
df_bronze = pd.read_csv(ARQ_BRONZE, sep=";")

# Merge com observações qualitativas do diário de bordo
df_merge = pd.merge(df_set, df_bronze[['data', 'observacao']], on='data', how='left')

print(f"Total de dias com refeição avaliados em setembro: {len(df_merge)}")
print(f"Dias regulares ('observado'): {len(df_merge[df_merge['status_dado'] == 'observado'])}")
print(f"Dias atípicos ('atipico'): {len(df_merge[df_merge['status_dado'] == 'atipico'])}")

# Visualização da tabela
df_merge[['data', 'dia_da_semana', 'consumo', 'status_dado', 'precipitation_sum', 'precipitation_hours', 'reservas', 'pred_ridge_c2', 'pred_hibrido']]


## 3. Diagnóstico Visual: Consumo Diário e Impacto Pluviométrico

Abaixo visualizamos a série temporal de setembro de 2026, destacando em vermelho os dias atípicos de inundação e em azul as operações regulares.


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

dias = df_merge['data'].values
consumo = df_merge['consumo'].values
chuva = df_merge['precipitation_sum'].values
status = df_merge['status_dado'].values

# Cores por status
cores = ['#d9534f' if s == 'atipico' else '#0275d8' for s in status]

# Gráfico de barras de consumo
barras = ax1.bar(range(len(dias)), consumo, color=cores, alpha=0.85, width=0.6, label='Consumo Real (Catraca)')
ax1.set_ylabel('Refeições Servidas', fontsize=12, fontweight='bold', color='#333333')
ax1.set_xticks(range(len(dias)))
ax1.set_xticklabels(dias, rotation=45, ha='right')
ax1.set_ylim(0, 260)

# Anotações dos dias extremos
for i, (val, s) in enumerate(zip(consumo, status)):
    if s == 'atipico':
        ax1.annotate(f"{int(val)} (Atípico)", 
                     (i, val), 
                     textcoords="offset points", 
                     xytext=(0, 8), 
                     ha='center', 
                     fontweight='bold', 
                     color='#d9534f')
    else:
        ax1.annotate(f"{int(val)}", 
                     (i, val), 
                     textcoords="offset points", 
                     xytext=(0, 5), 
                     ha='center', 
                     fontsize=9,
                     color='#333333')

# Eixo secundário: Precipitação em mm
ax2 = ax1.twinx()
linha_chuva = ax2.plot(range(len(dias)), chuva, color='#5bc0de', marker='o', linewidth=2.5, linestyle='--', label='Chuva Acumulada (mm)')
ax2.set_ylabel('Precipitação Acumulada (mm)', fontsize=12, fontweight='bold', color='#5bc0de')
ax2.grid(False)
ax2.set_ylim(0, 50)

plt.title('Série Temporal Diária: Consumo de Refeições vs. Volume Pluviométrico (Setembro/2026)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


## 4. Recálculo das Margens de Segurança (Conformal Prediction & Newsvendor)

A calibração original do artefato foi realizada na série walk-forward (2023 até meados de 2026). Agora, com o mês de setembro disponível, avaliamos o recálculo dos quantis de resíduo:

$$\text{Resíduo}_i = y_i - \hat{y}_i$$

* **Cenário A (Contaminação de Atípicos):** Se incluirmos os resíduos extremos dos dias de enchente ($y = 14$ com previsão $\hat{y} \approx 170$, gerando resíduo de $-156$), a variância dos resíduos explode, distorcendo a margem para cima em dias normais.
* **Cenário B (Série Causal Robusta):** Recalculamos as margens de erro utilizando a amostra ampliada de dias normais, isolando a perturbação climática catastrófica como regime de força maior.


In [ ]:
# Carga do histórico de predições walk-forward
WF_PATH = BASE_DIR / "dados" / "gold" / "predicoes_detalhadas_walk_forward.parquet"
df_wf = pd.read_parquet(WF_PATH)

# Filtrando modelo Ridge no Cenário 2 (Com Clima) e horizonte Semana_t1
df_wf_ridge = df_wf[(df_wf['algoritmo'] == 'Ridge') & (df_wf['cenario'] == 'Cenario_2_Com_Clima') & (df_wf['horizonte'] == 'Semana_t1')].copy()
residuos_historicos = (df_wf_ridge['y_real'] - df_wf_ridge['y_pred']).values

# Resíduos de setembro
df_merge['residuo_ridge'] = df_merge['consumo'] - df_merge['pred_ridge_c2']
residuos_set_normais = df_merge[df_merge['status_dado'] == 'observado']['residuo_ridge'].values
residuos_set_todos = df_merge['residuo_ridge'].values

# Concatenação das séries
residuos_expandidos_normais = np.concatenate([residuos_historicos, residuos_set_normais])
residuos_expandidos_com_atipicos = np.concatenate([residuos_historicos, residuos_set_todos])

# Função para cálculo dos quantis conformal (margem de segurança aditiva para proteção contra falta: y > y_pred)
def calcular_margens(residuos):
    res_positivos = residuos[residuos > 0] # Faltas potenciais (demanda real superou previsão)
    p75 = np.percentile(np.maximum(0, residuos), 75)
    p85 = np.percentile(np.maximum(0, residuos), 85)
    p90 = np.percentile(np.maximum(0, residuos), 90)
    
    # Newsvendor clássico: Cu = R$ 21.25 (custo social falta), Co = R$ 8.50 (custo sobra)
    # Razão crítica: Cu / (Cu + Co) = 21.25 / 29.75 = 0.714 (71.4%)
    p_nv = np.percentile(np.maximum(0, residuos), 71.4)
    return p75, p85, p90, p_nv

m_orig_75, m_orig_85, m_orig_90, m_orig_nv = calcular_margens(residuos_historicos)
m_rob_75, m_rob_85, m_rob_90, m_rob_nv = calcular_margens(residuos_expandidos_normais)
m_con_75, m_con_85, m_con_90, m_con_nv = calcular_margens(residuos_expandidos_com_atipicos)

df_margens = pd.DataFrame([
    {"Politica": "Conformal P75", "Original (Walk-Forward)": f"+{m_orig_75:.1f}", "Expandida (Dias Normais)": f"+{m_rob_75:.1f}", "Contaminada (Com Atípicos)": f"+{m_con_75:.1f}"},
    {"Politica": "Conformal P85", "Original (Walk-Forward)": f"+{m_orig_85:.1f}", "Expandida (Dias Normais)": f"+{m_rob_85:.1f}", "Contaminada (Com Atípicos)": f"+{m_con_85:.1f}"},
    {"Politica": "Conformal P90", "Original (Walk-Forward)": f"+{m_orig_90:.1f}", "Expandida (Dias Normais)": f"+{m_rob_90:.1f}", "Contaminada (Com Atípicos)": f"+{m_con_90:.1f}"},
    {"Politica": "Newsvendor (71.4%)", "Original (Walk-Forward)": f"+{m_orig_nv:.1f}", "Expandida (Dias Normais)": f"+{m_rob_nv:.1f}", "Contaminada (Com Atípicos)": f"+{m_con_nv:.1f}"}
])

print("📊 Comparativo das Margens de Segurança Recalibradas (Aditivo em Refeições):")
df_margens


## 5. Proposta da Regra de Mitigação Climática Operacional para a Defesa

Em dias normais, a adição de margem de segurança protege os estudantes contra o desabastecimento. No entanto, quando ocorrem alertas severos de chuva (precipitação contínua $\ge 10\text{ h}$ ou alerta formal da Defesa Civil), a prioridade operacional inverte-se para **evitar o desperdício massivo de alimentos**.

Definimos a **Previsão Operacional Resiliente**:
$$\hat{y}_{\text{operacional}} = \begin{cases} \hat{y}_{\text{híbrido}} + \text{Margem P85}, & \text{se } \text{alerta\_clima} = \text{Falso} \\ \min(\hat{y}_{\text{IA}}, \hat{y}_{\text{híbrido}}), & \text{se } \text{alerta\_clima} = \text{Verdadeiro} \end{cases}$$


In [ ]:
# Aplicação da regra de decisão no dataset de setembro
margem_p85 = float(m_rob_85)

# Definindo alerta de chuva extrema: precipitação acumulada > 15 mm OU horas de chuva >= 10 h
df_merge['alerta_chuva'] = (df_merge['precipitation_sum'] >= 15.0) | (df_merge['precipitation_hours'] >= 10.0) | (df_merge['status_dado'] == 'atipico')

# Previsões sob diferentes políticas
df_merge['pred_ia_pura'] = df_merge['pred_ridge_c2']
df_merge['pred_nutricionista'] = df_merge['reservas']
df_merge['pred_hibrido_puro'] = df_merge['pred_hibrido']
df_merge['pred_estatica_p85'] = df_merge['pred_ia_pura'] + margem_p85

# Margem prudencial ótima para o Híbrido (+12 ref Conformal da Qualificação)
MARGEM_PRUDENCIAL_HIBRIDO = 12.0

# Política Híbrida com Margem Prudencial (Fixa)
df_merge['pred_hibrido_margem_12'] = df_merge['pred_hibrido_puro'] + MARGEM_PRUDENCIAL_HIBRIDO

# Política Dinâmica Resiliente para a Defesa (Híbrido + Margem em dias normais; Contenção IA Pura sob alerta)
df_merge['pred_dinamica_resiliente'] = np.where(
    df_merge['alerta_chuva'],
    df_merge['pred_ia_pura'], # Em alerta severo de enchente, desativa margem aditiva e confia na redução climática da IA
    df_merge['pred_hibrido_puro'] + MARGEM_PRUDENCIAL_HIBRIDO # Em dias normais, protege o estudante contra falta
)

print("✅ Políticas operacionais calculadas com sucesso!")


## 6. Avaliação Comparativa de Métricas de Negócio (Sobra, Falta e Custos)

Avaliamos as políticas sob a ótica da gestão pública institucional:
* **Custo Unitário de Desperdício (Sobra):** R$ 8,50 por refeição não consumida;
* **Custo Social Unitário de Desabastecimento (Falta):** R$ 21,25 por aluno desassistido.


In [ ]:
CUSTO_SOBRA = 8.50
CUSTO_FALTA = 21.25

def avaliar_politica(df_subset, col_pred, nome):
    y_true = df_subset['consumo'].values
    y_pred = df_subset[col_pred].values
    
    erros = y_pred - y_true
    sobras = np.maximum(0, erros).sum()
    faltas = np.maximum(0, -erros).sum()
    mae = np.mean(np.abs(erros))
    rmse = np.sqrt(np.mean(erros ** 2))
    wape = (np.sum(np.abs(erros)) / np.sum(y_true)) * 100
    
    custo_sobra_rs = sobras * CUSTO_SOBRA
    custo_falta_rs = faltas * CUSTO_FALTA
    custo_total_rs = custo_sobra_rs + custo_falta_rs
    
    return {
        "Política": nome,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "WAPE (%)": f"{wape:.1f}%",
        "Sobra Total (ref)": int(sobras),
        "Falta Total (ref)": int(faltas),
        "Custo Sobras (R$)": f"R$ {custo_sobra_rs:,.2f}",
        "Custo Faltas (R$)": f"R$ {custo_falta_rs:,.2f}",
        "Custo Global (R$)": f"R$ {custo_total_rs:,.2f}"
    }

politicas = [
    ('pred_nutricionista', '1. Nutricionista Pura (Humano)'),
    ('pred_ia_pura', '2. IA Pura (Ridge C2)'),
    ('pred_hibrido_puro', '3. Modelo Híbrido 50/50 Puro'),
    ('pred_hibrido_margem_12', '4. Híbrido + Margem Prudencial (+12 ref)'),
    ('pred_dinamica_resiliente', '5. Política Dinâmica Resiliente (Defesa)')
]

# Avaliação nos 11 dias regulares
print("📋 --- CENÁRIO A: DIAS REGULARES (11 DIAS NORMAIS) ---")
res_regulares = [avaliar_politica(df_merge[df_merge['status_dado'] == 'observado'], col, nome) for col, nome in politicas]
df_res_reg = pd.DataFrame(res_regulares)
display(df_res_reg)

# Avaliação nos 5 dias atípicos de enchente
print("
⛈️ --- CENÁRIO B: DIAS ATÍPICOS DE ENCHENTE (5 DIAS EXTREMOS) ---")
res_atipicos = [avaliar_politica(df_merge[df_merge['status_dado'] == 'atipico'], col, nome) for col, nome in politicas]
df_res_atip = pd.DataFrame(res_atipicos)
display(df_res_atip)

# Avaliação no mês completo de setembro
print("
📅 --- CENÁRIO C: MÊS COMPLETO DE SETEMBRO (16 DIAS CONSOLIDADOS) ---")
res_completo = [avaliar_politica(df_merge, col, nome) for col, nome in politicas]
df_res_comp = pd.DataFrame(res_completo)
display(df_res_comp)


## 7. Exportação dos Resultados e Síntese para a Dissertação

Exportamos as tabelas geradas para a pasta `reports/`, garantindo rastreabilidade completa e integração com a interface operacional Streamlit.


In [ ]:
# Salvando os resultados consolidados
OUTPUT_DIR = BASE_DIR / "reports" / "defesa_retrospectiva_setembro"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_res_comp.to_csv(OUTPUT_DIR / "metricas_comparativas_setembro_2026.csv", index=False, sep=";")
df_margens.to_csv(OUTPUT_DIR / "recalculo_margens_seguranca_conformal.csv", index=False, sep=";")
df_merge.to_csv(OUTPUT_DIR / "previsoes_consolidadas_setembro_com_politica_resiliente.csv", index=False, sep=";")

print(f"✅ Artefatos exportados com sucesso em: {OUTPUT_DIR}")
